# Kémzy PersonaLive CUDA Proof v6

Run **all cells from top to bottom**. This notebook is the GitHub source of truth for the Kaggle proof: it clones the pinned PersonaLive commit, applies the committed compatibility patch, reuses already-mounted weights when available, and if they are missing downloads the required weights **inside Kaggle only**. It prepares the source/driving inputs, disables xFormers, and runs the real offline renderer.


In [ ]:
import os, sys, subprocess, json, pathlib, shutil, time
print('Python:', sys.executable)
print('Version:', sys.version.split()[0])
assert os.path.exists('/kaggle/working'), 'This notebook must run on Kaggle.'
print('Kaggle working directory: PASS')

In [ ]:
import subprocess, os, pathlib
ROOT='/kaggle/working/PersonaLive'
if os.path.isdir(ROOT):
    print('PersonaLive directory already exists; refreshing only if it is not the pinned commit.')
    r=subprocess.run(['git','-C',ROOT,'rev-parse','HEAD'],capture_output=True,text=True)
    if r.returncode==0 and r.stdout.strip()=='abdd112e01dcf7d89122c2e5efa29fcff0669740':
        print('Pinned PersonaLive checkout already present.')
    else:
        shutil.rmtree(ROOT)
if not os.path.isdir(ROOT):
    subprocess.run(['git','clone','https://github.com/GVCLab/PersonaLive.git',ROOT],check=True)
    subprocess.run(['git','-C',ROOT,'checkout','--detach','abdd112e01dcf7d89122c2e5efa29fcff0669740'],check=True)
head=subprocess.check_output(['git','-C',ROOT,'rev-parse','HEAD'],text=True).strip()
assert head=='abdd112e01dcf7d89122c2e5efa29fcff0669740', head
print('PersonaLive pinned commit:', head)

In [ ]:
import subprocess, pathlib, os
KEMZY='/kaggle/working/Kemzy-LiveAvatar'
if not os.path.isdir(KEMZY):
    subprocess.run(['git','clone','--branch','feature/backend-render-gateway','https://github.com/eneokonaniebiet/Kemzy-LiveAvatar.git',KEMZY],check=True)
else:
    subprocess.run(['git','-C',KEMZY,'fetch','origin','feature/backend-render-gateway'],check=True)
    subprocess.run(['git','-C',KEMZY,'reset','--hard','origin/feature/backend-render-gateway'],check=True)
print(subprocess.check_output(['git','-C',KEMZY,'rev-parse','HEAD'],text=True).strip())
patch=pathlib.Path(KEMZY)/'tools/personalive_compat_patch.py'
assert patch.is_file(), patch
subprocess.run([sys.executable,str(patch),ROOT],check=True)
print('Committed compatibility patch applied.')

In [ ]:
import importlib, subprocess, sys, types
# Keep Kaggle's protobuf/Torch/CUDA environment intact. MediaPipe's optional
# TensorFlow documentation import is stubbed below; missing packages are installed
# without dependencies so pip cannot silently downgrade protobuf.
if 'tensorflow.tools.docs' not in sys.modules:
    tf = types.ModuleType('tensorflow')
    tf_tools = types.ModuleType('tensorflow.tools')
    tf_docs = types.ModuleType('tensorflow.tools.docs')
    tf_docs.doc_controls = types.SimpleNamespace()
    tf_tools.docs = tf_docs
    tf.tools = tf_tools
    sys.modules['tensorflow'] = tf
    sys.modules['tensorflow.tools'] = tf_tools
    sys.modules['tensorflow.tools.docs'] = tf_docs
required={'mediapipe':'0.10.13','av':'18.1.0','decord':'0.6.0'}
missing=[]
for mod,want in required.items():
    try:
        m=importlib.import_module(mod)
        got=getattr(m,'__version__',None)
        print(mod, got)
        if got != want: missing.append(f'{mod}=={want}')
    except Exception as e:
        print(mod, 'import failed:', repr(e))
        missing.append(f'{mod}=={want}')
if missing:
    print('Installing without dependency resolution:', missing)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',*missing],check=True)
import torch, diffusers
print('Torch:',torch.__version__,'CUDA:',torch.cuda.is_available())
print('Diffusers:',diffusers.__version__)
assert torch.cuda.is_available(), 'CUDA GPU is required.'
print('GPU:',torch.cuda.get_device_name(0))
print('Protobuf intentionally left untouched by this cell.')


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
PR=Path(ROOT)/'pretrained_weights'
PR.mkdir(parents=True,exist_ok=True)
required_files=[
 'personalive/denoising_unet.pth','personalive/motion_encoder.pth','personalive/motion_extractor.pth','personalive/pose_guider.pth','personalive/reference_unet.pth','personalive/temporal_module.pth',
 'sd-vae-ft-mse/diffusion_pytorch_model.bin','sd-vae-ft-mse/config.json',
 'sd-image-variations-diffusers/image_encoder/pytorch_model.bin','sd-image-variations-diffusers/image_encoder/config.json',
 'sd-image-variations-diffusers/unet/diffusion_pytorch_model.bin','sd-image-variations-diffusers/unet/config.json','sd-image-variations-diffusers/model_index.json']
EXPECTED_SIZES={
 'personalive/denoising_unet.pth':4927015578,'personalive/motion_encoder.pth':246719031,'personalive/motion_extractor.pth':112545505,'personalive/pose_guider.pth':4351700,'personalive/reference_unet.pth':3438324340,'personalive/temporal_module.pth':1817903019,
 'sd-vae-ft-mse/diffusion_pytorch_model.bin':334707217,'sd-vae-ft-mse/config.json':547,
 'sd-image-variations-diffusers/image_encoder/pytorch_model.bin':1215993867,'sd-image-variations-diffusers/image_encoder/config.json':703,
 'sd-image-variations-diffusers/unet/diffusion_pytorch_model.bin':3438350225,'sd-image-variations-diffusers/unet/config.json':471,'sd-image-variations-diffusers/model_index.json':545}
def valid(p,rel):
    return p.is_file() and p.stat().st_size==EXPECTED_SIZES[rel]
def find_file(rel):
    name=Path(rel).name
    for base in [Path('/kaggle/input'),Path('/kaggle/working')]:
        if not base.exists(): continue
        for p in base.rglob(name):
            if valid(p,rel): return p
    return None
missing=[]
for rel in required_files:
    dst=PR/rel
    if valid(dst,rel): continue
    src=find_file(rel)
    if src is not None:
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.exists() or dst.is_symlink(): dst.unlink()
        dst.symlink_to(src)
        print('mounted:',rel,'<-',src)
    else: missing.append(rel)
if missing:
    print('Mounted dataset does not contain all PersonaLive weights.')
    print('Missing:',missing)
    print('SELF-HEALING: downloading ONLY the missing PersonaLive/base/VAE files inside Kaggle (never to the phone)...')
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','huggingface_hub'],check=True)
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='huaichang/PersonaLive',local_dir=str(PR),allow_patterns=required_files)
    missing=[]
    bad=[]
    for rel in required_files:
        p=PR/rel
        if not p.is_file(): missing.append(rel)
        elif p.stat().st_size != EXPECTED_SIZES[rel]: bad.append(f'{rel}: got {p.stat().st_size}, expected {EXPECTED_SIZES[rel]}')
    if missing or bad:
        raise RuntimeError('PersonaLive weight self-heal failed. Missing:\n'+'\n'.join(missing)+'\nBad sizes:\n'+'\n'.join(bad))
print('All required PersonaLive/base/VAE files are present and exact-size verified.')
for rel in required_files: print(f'{rel}: {(PR/rel).stat().st_size} bytes')


In [ ]:
from pathlib import Path
import shutil
demo=Path(ROOT)/'demo'; demo.mkdir(exist_ok=True)
def first_matching(names):
    for base in [Path('/kaggle/input'),Path('/kaggle/working')]:
        if base.exists():
            for n in names:
                for p in base.rglob(n):
                    if p.is_file() and '/PersonaLive/' not in str(p): return p
    return None
img=Path('/kaggle/input/datasets/seravellenyrovalen/kemzyphoto/7322a267d7716f19b6792bf610fb0961.jpg')
if not img.is_file(): img=first_matching(['*.jpg','*.jpeg','*.png'])
vid=first_matching(['driving_video.mp4'])
if vid is None:
    vid=first_matching(['*.mp4'])
assert img is not None and img.is_file(), 'Reference image not found in Kaggle input.'
assert vid is not None and vid.is_file(), 'Driving video not found in Kaggle input.'
shutil.copy2(img,demo/'ref_img.jpg')
shutil.copy2(vid,demo/'driving_video.mp4')
print('Reference:',img, img.stat().st_size)
print('Driving:',vid, vid.stat().st_size)

In [ ]:
import sys, pathlib, torch
sys.path.insert(0,ROOT)
from src.models.motion_encoder.encoder import MotEncoder
m=MotEncoder().eval()
pe=m.pe
print('MotionEncoder PE shape:',tuple(pe.shape))
assert tuple(pe.shape)==(1,32,16), tuple(pe.shape)
print('MotionEncoder compatibility: PASS')

In [ ]:
import json, urllib.request, urllib.error

# Secure Kaggle-only diagnostic assistant. The API key is read from Kaggle Secrets
# and is never printed, committed, or included in logs.
OPENAI_DIAGNOSTIC_READY=False
try:
    from kaggle_secrets import UserSecretsClient
    _secrets=UserSecretsClient()
    OPENAI_API_KEY=_secrets.get_secret('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        raise RuntimeError('OPENAI_API_KEY is empty.')
    OPENAI_DIAGNOSTIC_READY=True
    print('OpenAI diagnostic connection: SECRET FOUND')
except Exception as e:
    print('OpenAI diagnostic unavailable:',type(e).__name__,str(e))

def ask_chatgpt_diagnostic(error_text, context=''):
    if not OPENAI_DIAGNOSTIC_READY:
        return None
    prompt=(
        'You are the debugging engineer for the Kémzy PersonaLive Kaggle GPU proof. '
        'Analyze the supplied failure and give the smallest concrete source-code fix. '
        'Do not suggest downloading multi-GB model files to a phone. Keep the pinned '
        'PersonaLive commit unchanged. Prefer fixes that can be committed to GitHub. '
        'Do not expose secrets.\n\nContext:\n'+context+'\n\nTraceback:\n'+error_text
    )
    body=json.dumps({'model':'gpt-5.6-luna','input':prompt,'store':False}).encode('utf-8')
    req=urllib.request.Request(
        'https://api.openai.com/v1/responses',data=body,method='POST',
        headers={'Authorization':'Bearer '+OPENAI_API_KEY,'Content-Type':'application/json'})
    try:
        with urllib.request.urlopen(req,timeout=120) as r:
            data=json.loads(r.read().decode('utf-8'))
        text=data.get('output_text')
        if not text:
            parts=[]
            for item in data.get('output',[]):
                for c in item.get('content',[]):
                    if c.get('type')=='output_text': parts.append(c.get('text',''))
            text='\n'.join(parts)
        print('\n===== OpenAI diagnostic =====\n'+(text or 'No text returned.')+'\n===== End diagnostic =====')
        return text
    except Exception as e:
        print('OpenAI diagnostic request failed:',type(e).__name__,str(e))
        return None


In [ ]:
import subprocess, os, pathlib, time, traceback
out=pathlib.Path(ROOT)/'results'
if out.exists(): shutil.rmtree(out)
cmd=[sys.executable,'inference_offline.py','--config','configs/prompts/personalive_offline.yaml','--name','kemzy_cuda_proof_v6','-W','512','-H','512','-L','4','--device','cuda','--reference_image',str(demo/'ref_img.jpg'),'--driving_video',str(demo/'driving_video.mp4')]
print('Running real PersonaLive inference with xFormers disabled by the committed patch...')
started=time.time()
run=subprocess.run(cmd,cwd=ROOT,text=True,capture_output=True)
print(run.stdout)
if run.returncode!=0:
    print(run.stderr)
    diagnostic=ask_chatgpt_diagnostic(
        run.stdout+'\n'+run.stderr,
        context='GitHub commit: a38ea8eb0e7bd2e1e43862ffe5980e81915c4df4; PersonaLive commit: abdd112e01dcf7d89122c2e5efa29fcff0669740; GPU: '+torch.cuda.get_device_name(0)
    )
    raise RuntimeError('PersonaLive inference failed. OpenAI diagnostic was printed above.' if diagnostic else 'PersonaLive inference failed; see captured stderr above.')
print('Inference seconds:',round(time.time()-started,2))


In [ ]:
from pathlib import Path
mp4s=sorted((Path(ROOT)/'results').rglob('*.mp4'))
print('Generated videos:',[(str(p),p.stat().st_size) for p in mp4s])
assert mp4s and all(p.stat().st_size>1000 for p in mp4s), 'No non-empty generated PersonaLive video found.'
proof=Path('/kaggle/working/kemzy_personalive_cuda_proof_v6.txt')
proof.write_text('PersonaLive CUDA proof PASS\ncommit=abdd112e01dcf7d89122c2e5efa29fcff0669740\nrenderer_output='+str(mp4s[0])+'\nsize='+str(mp4s[0].stat().st_size)+'\n',encoding='utf-8')
print(proof.read_text())

In [ ]:
# Optional backend smoke-test: import the Kémzy gateway without starting a second long-lived server.
import sys
sys.path.insert(0,KEMZY)
import backend.renderer.personalive_server as gateway
gateway.APP_ARGS=gateway.build_args()
print('Gateway import: PASS')
print('Gateway acceleration:',gateway.APP_ARGS.acceleration)
assert gateway.APP_ARGS.acceleration=='none'
print('Backend configuration: PASS — ACCELERATION=none')